In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
train1 = pd.read_csv('../data/case1/train1.csv')
test1 = pd.read_csv('../data/case1/test1.csv')

In [4]:
def add_feature_combinations(df):
    df = df.copy()
    
    df['speed_krpm'] = df['speed_rpm'] / 1000.0
    df['delta_p'] = df['outlet_pressure_bar'] - df['inlet_pressure_bar']
    
    df['speed_x_delta_p'] = df['speed_krpm'] * df['delta_p']      # interaction
    df['power_per_speed'] = df['power_w'] / (df['speed_krpm'] + 1e-6)  # power ratio
    df['speed_sq'] = df['speed_krpm'] ** 2                         # quadratic speed
    df['delta_p_sq'] = df['delta_p'] ** 2                          # quadratic pressure
    return df

In [5]:
train1_fe = add_feature_combinations(train1)
test1_fe = add_feature_combinations(test1)

In [6]:
# model A: simple raw features
raw_cols = ['speed_krpm', 'power_w', 'delta_p']
model_raw = LinearRegression().fit(train1_fe[raw_cols], train1_fe['flow_m3_h'])
pred_raw = model_raw.predict(test1_fe[raw_cols])
rmse_raw = np.sqrt(mean_squared_error(test1_fe['flow_m3_h'], pred_raw))

In [7]:
# model B: upgraded features with combinations
fe_cols = ['speed_krpm', 'power_w', 'delta_p', 'speed_x_delta_p', 'power_per_speed', 'speed_sq', 'delta_p_sq']
model_fe = LinearRegression().fit(train1_fe[fe_cols], train1_fe['flow_m3_h'])
pred_fe = model_fe.predict(test1_fe[fe_cols])
rmse_fe = np.sqrt(mean_squared_error(test1_fe['flow_m3_h'], pred_fe))

In [11]:
print(f"Task 1 (Raw Features)      -> Test RMSE: {rmse_raw:.4f} m³/h")
print(f"Task 1 (With Combinations) -> Test RMSE: {rmse_fe:.4f} m³/h")

Task 1 (Raw Features)      -> Test RMSE: 3.5363 m³/h
Task 1 (With Combinations) -> Test RMSE: 1.0788 m³/h
